**<h1 align="center">Rerankers</h1>**

This notebook trains learned reranking models on ESCO-augmented SBERT candidate sets. Logistic regression and XGBoost are fitted on the validation split to combine SBERT similarity with multiple ESCO-derived features.

The trained rerankers are then applied to validation and test splits to produce reranked candidate lists. Performance is evaluated using the same retrieval metrics as earlier stages, and feature importance is analyzed to support interpretability.

# Table of Contents
* [1. Imports & Setup](#chapter1)
* [2. Load Data](#chapter2)
* [3. Evaluation metrics](#chapter3)
* [4. Reranker](#chapter4)
    * [4.1. Logistic Regression](#sub-section-4_1)
    * [4.2. XGBoost](#sub-section-4_2)
    * [4.3. ESCO contributions](#sub-section-4_3)

<a class="anchor" id="chapter1"></a>

# 1. Imports & Setup

</a>

In [ ]:
import sys
import json
import math
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

In [ ]:
# Project root (assumes notebook is in /Notebook)
PROJECT_ROOT = Path("..").resolve()

# Allow imports from project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from thesis_utility import save_json

In [ ]:
DATA_DIR = PROJECT_ROOT / "Data" / "TopK"
OUT_DIR = PROJECT_ROOT / "Results" / "Rerankers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME = datetime.now().strftime("%m%d%Y_%H%M")

<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [ ]:
# Load top-K candidate sets produced in notebook 05
val_topk_feat_off = pd.read_parquet(DATA_DIR / "val_topk_feat_off.parquet")
test_topk_feat_off = pd.read_parquet(DATA_DIR / "test_topk_feat_off.parquet")

In [ ]:
# Ensure occupation URIs are strings
for df in [val_topk_feat_off, test_topk_feat_off]:
    df["candidate_occ_uri"] = df["candidate_occ_uri"].astype(str)
    df["true_occ_uri"] = df["true_occ_uri"].astype(str)

In [ ]:
print("Loaded val:", val_topk_feat_off.shape)
print("Loaded test:", test_topk_feat_off.shape)
display(val_topk_feat_off.head(2))

Loaded val: (4424450, 14)
Loaded test: (4432150, 14)


,person_id,t,true_occ_uri,candidate_occ_uri,rank,sbert_score,overlap_count,occ_coverage,cv_coverage,jaccard,idf_overlap_sum,idf_overlap_mean,occ_avg_skill_idf,idf_overlap_sum_parent
0,5,1,http://data.europa.eu/esco/occupation/044d78cc...,http://data.europa.eu/esco/occupation/045f05f2...,1,0.762878,14,1.00,1.0,1.00000,86.898468,6.207034,6.207034,86.898468
1,5,1,http://data.europa.eu/esco/occupation/044d78cc...,http://data.europa.eu/esco/occupation/3469cbcc...,2,0.699551,7,0.28,0.5,0.21875,46.250938,6.607277,5.857293,46.250938


<a class="anchor" id="chapter3"></a>

# 3. Evaluation metrics

</a>

In [ ]:
# Compute ranking metrics from reranked candidate lists
def eval_metrics(df, rank_col="new_rank", k_list=(1, 3, 5, 10, 20)):
    df = df.copy()
    df["is_true"] = (df["candidate_occ_uri"] == df["true_occ_uri"]).astype(int)

    # Retrieve rank of the true occupation for each (person_id, t) query
    true_ranks = df[df["is_true"] == 1].groupby(["person_id", "t"])[rank_col].min()

    # Total number of queries
    num_queries = df.groupby(["person_id", "t"]).ngroups
    out = {}

    # Compute Recall@K (Accuracy@K and Precision@K also computed)
    for K in k_list:
        hit = (true_ranks <= K).sum()
        recall_k = float(hit / num_queries)

        # Under a single-relevance setting, Accuracy@K is equivalent to Recall@K
        out[f"accuracy@{K}"] = recall_k
        out[f"recall@{K}"] = recall_k

        # Precision@K = 1/K if the true item is retrieved within top-K, else 0
        out[f"precision@{K}"] = float(((true_ranks <= K) * (1.0 / K)).sum() / num_queries)

    # Use K=10 as the default cutoff for MRR (or the largest K if 10 is unavailable)
    K_mrr = 10 if 10 in k_list else max(k_list)

    rr = true_ranks.copy()
    rr = rr[rr <= K_mrr]
    mrr = float((1.0 / rr).sum() / num_queries)
    out[f"mrr@{K_mrr}"] = mrr

    # Compute nDCG@K for a single relevant item per query
    def ndcg_single(rank, K):
        return (1.0 / math.log2(rank + 1)) if rank <= K else 0.0

    for K in k_list:
        nd = sum(ndcg_single(r, K) for r in true_ranks.dropna())
        out[f"ndcg@{K}"] = float(nd / num_queries)

    # MAP is included for completeness; under single relevance it is equivalent to MRR
    out[f"map@{K_mrr}"] = mrr

    return out

<a class="anchor" id="chapter4"></a>

# 4. Reranker

</a>

In [ ]:
QUERY_COLS = ["person_id", "t"]

In [ ]:
ALL_FEATURES = [
    "sbert_score", # retrieval signal
    "overlap_count",
    "occ_coverage",
    "cv_coverage",
    "jaccard",
    "idf_overlap_sum",
    "idf_overlap_mean",
    "occ_avg_skill_idf",
    "idf_overlap_sum_parent",
]

In [ ]:
# Prepare feature matrix (X) and labels (y) for reranker training
def make_reranker_xy(df_topk: pd.DataFrame, features=ALL_FEATURES):
    df = df_topk.copy()

    # Create binary label: 1 if candidate is the true occupation, else 0
    df["label"] = (df["candidate_occ_uri"] == df["true_occ_uri"]).astype(int)

    # Ensure all required feature columns are present
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise ValueError(f"Missing reranker features: {missing}")

    # Build feature matrix (handle NaN and infinite values)
    X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0.0).values

    # Build feature matrix (handle NaN and infinite values)
    y = df["label"].astype(int).values
    return df, X, y

<a class="anchor" id="sub-section-4_1"></a>

## 4.1. Logistic Regression

</a>

In [ ]:
# Train
df_val_lr, X_val_lr, y_val_lr = make_reranker_xy(val_topk_feat_off, features=ALL_FEATURES)

In [ ]:
logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        solver="liblinear",
        max_iter=500,
        class_weight="balanced",
        random_state=42,
    ))
])

logreg_model.fit(X_val_lr, y_val_lr)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=500,
                                    random_state=42, solver='liblinear'))])

In [ ]:
# Apply to val
df_val_apply, X_val_apply, _ = make_reranker_xy(val_topk_feat_off, features=ALL_FEATURES)
val_logreg_scored = df_val_apply.copy()
val_logreg_scored["rerank_score"] = logreg_model.predict_proba(X_val_apply)[:, 1]
val_logreg_scored["new_rank"] = val_logreg_scored.groupby(["person_id", "t"])["rerank_score"].rank(ascending=False, method="first")

In [ ]:
# Apply to test
df_test_apply, X_test_apply, _ = make_reranker_xy(test_topk_feat_off, features=ALL_FEATURES)
test_logreg_scored = df_test_apply.copy()
test_logreg_scored["rerank_score"] = logreg_model.predict_proba(X_test_apply)[:, 1]
test_logreg_scored["new_rank"] = test_logreg_scored.groupby(["person_id", "t"])["rerank_score"].rank(ascending=False, method="first")

In [ ]:
# Evaluate
val_logreg_metrics = eval_metrics(val_logreg_scored, rank_col="new_rank", k_list=(1,3,5,10,20))
test_logreg_metrics = eval_metrics(test_logreg_scored, rank_col="new_rank", k_list=(1,3,5,10,20))

In [ ]:
clf = logreg_model.named_steps["clf"]

coef_df = (
    pd.DataFrame({
        "feature": ALL_FEATURES,
        "weight": clf.coef_.ravel()
    })
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)

intercept = float(clf.intercept_[0])

In [ ]:
print("Intercept:", intercept)
display(coef_df)

Intercept: -0.5039221160073452


,feature,weight
0,occ_coverage,0.773929
1,idf_overlap_sum_parent,0.695473
2,idf_overlap_sum,0.695473
3,cv_coverage,0.452483
4,occ_avg_skill_idf,0.336722
5,idf_overlap_mean,0.182148
6,sbert_score,0.062292
7,jaccard,-0.511147
8,overlap_count,-1.488748


In [ ]:
#save_json(val_logreg_metrics, f"ESCO_logreg_reranker_offshelf_val_{TIME}.json", out_dir=OUT_DIR)
#save_json(test_logreg_metrics, f"ESCO_logreg_reranker_offshelf_test_{TIME}.json", out_dir=OUT_DIR)

In [ ]:
# Save coefficients for interpretability
#coef_df.to_csv(f"{OUT_DIR}/ESCO_logreg_coefficients_offshelf_{TIME}.csv", index=False)
#save_json({"intercept": intercept}, f"ESCO_logreg_intercept_offshelf_{TIME}.json", out_dir=OUT_DIR)

<a class="anchor" id="sub-section-4_2"></a>

## 4.2. XGBoost

</a>

In [ ]:
# Train
df_val_xgb, X_val_xgb, y_val_xgb = make_reranker_xy(val_topk_feat_off, features=ALL_FEATURES)

In [ ]:
n_pos = int((y_val_xgb == 1).sum())
n_neg = int((y_val_xgb == 0).sum())
scale_pos_weight = n_neg / max(n_pos, 1)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300, # number of trees 
    learning_rate=0.05, # smaller step size for more stable learning
    max_depth=4, # shallow trees to limit model complexity
    min_child_weight=1,
    subsample=0.8, # shallow trees to limit model complexity
    colsample_bytree=0.8, # feature sampling
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

xgb_model.fit(X_val_xgb, y_val_xgb)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)

In [ ]:
# Apply to val
df_val_apply, X_val_apply, _ = make_reranker_xy(val_topk_feat_off, features=ALL_FEATURES)
val_xgb_scored = df_val_apply.copy()
val_xgb_scored["rerank_score"] = xgb_model.predict_proba(X_val_apply)[:, 1]
val_xgb_scored["new_rank"] = val_xgb_scored.groupby(["person_id", "t"])["rerank_score"].rank(ascending=False, method="first")

In [ ]:
# Apply to test
df_test_apply, X_test_apply, _ = make_reranker_xy(test_topk_feat_off, features=ALL_FEATURES)
test_xgb_scored = df_test_apply.copy()
test_xgb_scored["rerank_score"] = xgb_model.predict_proba(X_test_apply)[:, 1]
test_xgb_scored["new_rank"] = test_xgb_scored.groupby(["person_id", "t"])["rerank_score"].rank(ascending=False, method="first")

In [ ]:
# Evaluate
val_xgb_metrics = eval_metrics(val_xgb_scored, rank_col="new_rank", k_list=(1,3,5,10,20))
test_xgb_metrics = eval_metrics(test_xgb_scored, rank_col="new_rank", k_list=(1,3,5,10,20))

In [ ]:
score_dict = xgb_model.get_booster().get_score(importance_type="gain")

rows = []
for i, feat in enumerate(ALL_FEATURES):
    rows.append({"feature": feat, "importance_gain": float(score_dict.get(f"f{i}", 0.0))})

xgb_imp_gain = pd.DataFrame(rows).sort_values("importance_gain", ascending=False).reset_index(drop=True)
display(xgb_imp_gain)

,feature,importance_gain
0,idf_overlap_mean,18794.835938
1,occ_coverage,15287.700195
2,jaccard,8409.945312
3,occ_avg_skill_idf,6171.469238
4,idf_overlap_sum_parent,5318.838867
5,idf_overlap_sum,3764.195801
6,cv_coverage,3449.707520
7,overlap_count,2236.645752
8,sbert_score,1692.693604


In [ ]:
#save_json(val_xgb_metrics, f"ESCO_xgb_reranker_offshelf_val_{TIME}.json", out_dir=OUT_DIR)
#save_json(test_xgb_metrics, f"ESCO_xgb_reranker_offshelf_test_{TIME}.json", out_dir=OUT_DIR)

In [ ]:
# Save importance gains for interpretability
#xgb_imp_gain.to_csv(f"{OUT_DIR}/xgb_feature_importance_gain_offshelf_{TIME}.csv", index=False)

<a class="anchor" id="sub-section-4_3"></a>

## 4.3. ESCO contributions

</a>

SBERT-only rerankers: Logistic Regression & XGBoost to see ESCO contributions

In [ ]:
SBERT_ONLY = ["sbert_score"]
K_LIST = (10,)

In [ ]:
# 1) Logistic Regression (SBERT-only)

# Train on val
df_val_lr, X_val_lr, y_val_lr = make_reranker_xy(val_topk_feat_off, features=SBERT_ONLY)

logreg_sbert = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        solver="liblinear",
        max_iter=500,
        class_weight="balanced",
        random_state=42,
    ))
])
logreg_sbert.fit(X_val_lr, y_val_lr)

# Apply to val
df_val_apply, X_val_apply, _ = make_reranker_xy(val_topk_feat_off, features=SBERT_ONLY)
val_logreg_sbert = df_val_apply
val_logreg_sbert["rerank_score"] = logreg_sbert.predict_proba(X_val_apply)[:, 1]
val_logreg_sbert["new_rank"] = val_logreg_sbert.groupby(["person_id", "t"])["rerank_score"].rank(
    ascending=False, method="first"
)

# Apply to test
df_test_apply, X_test_apply, _ = make_reranker_xy(test_topk_feat_off, features=SBERT_ONLY)
test_logreg_sbert = df_test_apply
test_logreg_sbert["rerank_score"] = logreg_sbert.predict_proba(X_test_apply)[:, 1]
test_logreg_sbert["new_rank"] = test_logreg_sbert.groupby(["person_id", "t"])["rerank_score"].rank(
    ascending=False, method="first"
)

val_rec_logreg_sbert = float(eval_metrics(val_logreg_sbert, rank_col="new_rank", k_list=K_LIST)["recall@10"])
test_rec_logreg_sbert = float(eval_metrics(test_logreg_sbert, rank_col="new_rank", k_list=K_LIST)["recall@10"])

In [ ]:
# 2) XGBoost (SBERT-only)

# Train on val
df_val_xgb, X_val_xgb, y_val_xgb = make_reranker_xy(val_topk_feat_off, features=SBERT_ONLY)

n_pos = int((y_val_xgb == 1).sum())
n_neg = int((y_val_xgb == 0).sum())
scale_pos_weight = n_neg / max(n_pos, 1)

xgb_sbert = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)
xgb_sbert.fit(X_val_xgb, y_val_xgb)

# Apply to val
df_val_apply, X_val_apply, _ = make_reranker_xy(val_topk_feat_off, features=SBERT_ONLY)
val_xgb_sbert = df_val_apply
val_xgb_sbert["rerank_score"] = xgb_sbert.predict_proba(X_val_apply)[:, 1]
val_xgb_sbert["new_rank"] = val_xgb_sbert.groupby(["person_id", "t"])["rerank_score"].rank(
    ascending=False, method="first"
)

# Apply to val
df_test_apply, X_test_apply, _ = make_reranker_xy(test_topk_feat_off, features=SBERT_ONLY)
test_xgb_sbert = df_test_apply
test_xgb_sbert["rerank_score"] = xgb_sbert.predict_proba(X_test_apply)[:, 1]
test_xgb_sbert["new_rank"] = test_xgb_sbert.groupby(["person_id", "t"])["rerank_score"].rank(
    ascending=False, method="first"
)

val_rec_xgb_sbert = float(eval_metrics(val_xgb_sbert, rank_col="new_rank", k_list=K_LIST)["recall@10"])
test_rec_xgb_sbert = float(eval_metrics(test_xgb_sbert, rank_col="new_rank", k_list=K_LIST)["recall@10"])

In [ ]:
# Comparison table: SBERT-only vs ESCO+SBERT rerankers (Recall@10 + deltas)

val_rec_logreg_full = float(val_logreg_metrics["recall@10"])
test_rec_logreg_full = float(test_logreg_metrics["recall@10"])
val_rec_xgb_full = float(val_xgb_metrics["recall@10"])
test_rec_xgb_full = float(test_xgb_metrics["recall@10"])

In [ ]:
comparison_table = pd.DataFrame([
    {
        "model": "LogReg",
        "val_recall@10_sbert_only": val_rec_logreg_sbert,
        "val_recall@10_full": val_rec_logreg_full,
        "val_delta": val_rec_logreg_full - val_rec_logreg_sbert,
        "test_recall@10_sbert_only": test_rec_logreg_sbert,
        "test_recall@10_full": test_rec_logreg_full,
        "test_delta": test_rec_logreg_full - test_rec_logreg_sbert,
    },
    {
        "model": "XGBoost",
        "val_recall@10_sbert_only": val_rec_xgb_sbert,
        "val_recall@10_full": val_rec_xgb_full,
        "val_delta": val_rec_xgb_full - val_rec_xgb_sbert,
        "test_recall@10_sbert_only": test_rec_xgb_sbert,
        "test_recall@10_full": test_rec_xgb_full,
        "test_delta": test_rec_xgb_full - test_rec_xgb_sbert,
    },
])

comparison_table = comparison_table.sort_values("test_delta", ascending=False).reset_index(drop=True)
display(comparison_table)

,model,val_recall@10_sbert_only,val_recall@10_full,val_delta,test_recall@10_sbert_only,test_recall@10_full,test_delta
0,XGBoost,0.110262,0.175355,0.065093,0.107826,0.171779,0.063953
1,LogReg,0.109776,0.143057,0.033281,0.108514,0.142166,0.033652


In [ ]:
# Save VAL reranker scores for EDA (small, stable files)
keep_cols = ["person_id", "t", "candidate_occ_uri", "true_occ_uri", "rerank_score"]

#val_logreg_scored[keep_cols].to_parquet(f"{OUT_DIR}/val_logreg_scored.parquet", index=False)
#val_xgb_scored[keep_cols].to_parquet(f"{OUT_DIR}/val_xgb_scored.parquet", index=False)